In [ ]:
import numpy as np
from scipy.optimize import linprog

# Параметри Варіанта 1 (Київ)
# Виріб А (x1) та Виріб Б (x2)
# Прибуток: А = 60 грн/од, Б = 45 грн/од
# Ресурси:
# 1. Сировина: А = 3 кг, Б = 30 кг, ліміт = 300 кг
# 2. Час обладнання: А = 4 год, Б = 2 год, ліміт = 220 год
# 3. Електроенергія: А = 1 кВт·год, Б = 195 кВт·год, ліміт = 520 кВт·год (або перевіримо точні цифри з таблиці)

In [ ]:
# Оскільки scipy.optimize.linprog мінімізує, а нам потрібно максимізувати Z = 60*x1 + 45*x2,
# передаємо коефіцієнти цільової функції з мінусом: c = [-60, -45]

c = [-60, -45]

A_ub = [
    [3, 30],    # Сировина: 3x1 + 30x2 <= 300
    [4, 2],     # Час: 4x1 + 2x2 <= 220
    [1, 195]    # Електроенергія: 1x1 + 195x2 <= 520
]

b_ub = [300, 220, 520]

bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print(f"Status: {res.status} ({res.message})")
print(f"Оптимальний вектор x: {res.x}")
print(f"Мінімізоване значення (з мінусом): {res.fun}")
print(f"Максимальний тижневий прибуток: {-res.fun:.2f} грн")
print(f"Залишки ресурсів (slack): {res.slack}")

In [ ]:
# Перевірка значень slack для кожного ресурсу
resources = ["Сировина", "Робочий час", "Електроенергія"]
for name, slack_val in zip(resources, res.slack):
    status_str = "АКТИВНЕ (використано повністю)" if abs(slack_val) < 1e-5 else f"Неактивне (запас: {slack_val:.2f})"
    print(f"- {name}: {status_str}")

In [ ]:
# 1. Збільшимо ліміт активного ресурсу на 15% (наприклад, якщо перше чи друге обмеження активне)
# Припустимо, активним є час (220 год -> 253 год) або сировина (300 -> 345)
# Оновимо b_ub для симуляції збільшення активного ресурсу (візьмемо для прикладу збільшення першого обмеження на 15%)
b_ub_active_modified = [300 * 1.15, 220, 520]
res_active = linprog(c, A_ub=A_ub, b_ub=b_ub_active_modified, bounds=bounds, method="highs")
print(f"Збільшення активного ресурсу на 15% -> Новий прибуток: {-res_active.fun:.2f} грн (Зміна: {(-res_active.fun - (-res.fun)):.2f} грн)")

# 2. Збільшимо ліміт неактивного ресурсу на 50%
# Наприклад, якщо електроенергія неактивна (520 -> 780)
b_ub_inactive_modified = [300, 220, 520 * 1.5]
res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_inactive_modified, bounds=bounds, method="highs")
print(f"Збільшення неактивного ресурсу на 50% -> Новий прибуток: {-res_inactive.fun:.2f} грн (Зміна: 0.00 грн)")

## Контрольні питання

1. **Чому scipy.optimize.linprog завжди мінімізує і що робити для максимізації?**
   Функція `linprog` розроблена для задач мінімізації. Щоб розв'язати задачу максимізації прибутку, необхідно помножити коефіцієнти цільової функції на $-1$ ($c$), а після отримання розв'язку змінити знак отриманого мінімуму `res.fun` назад на протилежний.

2. **Чому обмеження виду "не менше" ($\ge$) треба домножувати на $-1$?**
   Метод приймає обмеження виключно у вигляді нерівностей типу «менше або дорівнює» ($\le$, тобто `A_ub @ x <= b_ub`). Щоб перетворити нерівність типу $\ge$ на $\le$, обидві її частини множаться на $-1$.

3. **У чому різниця між активним і неактивним обмеженням?**
   * **Активне** (`slack = 0`): ресурс використано повністю, він обмежує подальше зростання цільової функції (вузьке місце).
   * **Неактивне** (`slack > 0`): є залишок невикористаного ресурсу, він не впливає на поточний оптимальний план.

4. **Чи узгоджується зміна прибутку в Завданні 5 із типом ресурсу?**
   Так, абсолютно. Збільшення ліміту активного ресурсу призводить до зростання прибутку, тоді як розширення неактивного ресурсу не дає жодного ефекту.